# Target variable visualization (standalone example)

This notebook creates a small synthetic price series, builds a simple target variable (Up / Down / Sideways) using dynamic thresholds based on recent volatility, and shows two visualizations suitable for non-technical audiences: a static annotated time-series and an interactive Plotly timeline. Run all cells in order.

## How the target is created (plain language)

We look ahead a fixed number of periods and compare the future percentage change to a moving volatility-based threshold.
- If the future change is higher than the positive threshold → 'Up'
- If the future change is lower than the negative threshold → 'Down'
- Otherwise → 'Sideways'

The thresholds move with recent volatility so they adapt when the market gets choppy or calm.

In [2]:
# Standard imports for the example
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set(style='whitegrid')

In [3]:
# Generate synthetic price data (sinusoid + noise)
np.random.seed(0)
n = 300
t = pd.date_range(end=pd.Timestamp.today(), periods=n, freq='H')
price = 100 + np.sin(np.linspace(0, 12 * np.pi, n)) * 0.5 + np.random.normal(scale=2.0, size=n).cumsum() * 0.1
df = pd.DataFrame({'Close': price}, index=t)

# Parameters for target creation
LOOKBACK = 20  # periods to estimate volatility (hours)
HORIZON = 5    # look-ahead periods to define the future move
MULTIPLIER = 1.0  # threshold multiplier of recent std

# Compute future pct change and rolling volatility
df['FutureReturn'] = df['Close'].shift(-HORIZON) / df['Close'] - 1
df['Vol'] = df['Close'].pct_change().rolling(window=LOOKBACK, min_periods=1).std()
df['UpperThreshold'] = MULTIPLIER * df['Vol']
df['LowerThreshold'] = -MULTIPLIER * df['Vol']

# Create target: 1 = Up, 0 = Down, 2 = Sideways
def make_target(row):
    fr = row['FutureReturn']
    if pd.isna(fr) or pd.isna(row['UpperThreshold']):
        return np.nan
    if fr > row['UpperThreshold']:
        return 1
    if fr < row['LowerThreshold']:
        return 0
    return 2

df['Target'] = df.apply(make_target, axis=1)
df = df.dropna(subset=['Target']).copy()  # drop tail rows without future label
df['TargetLabel'] = df['Target'].map({0: 'Down', 1: 'Up', 2: 'Sideways'})

# Show head
df.tail()

C:\Users\nicol\AppData\Local\Temp\ipykernel_37412\3864835007.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  t = pd.date_range(end=pd.Timestamp.today(), periods=n, freq='H')


,Close,FutureReturn,Vol,UpperThreshold,LowerThreshold,Target,TargetLabel
2025-10-20 04:20:47.602642,100.715539,0.005604,0.002564,0.002564,-0.002564,1.0,Up
2025-10-20 05:20:47.602642,100.649487,0.007024,0.002145,0.002145,-0.002145,1.0,Up
2025-10-20 06:20:47.602642,101.147140,0.003814,0.002294,0.002294,-0.002294,1.0,Up
2025-10-20 07:20:47.602642,100.978138,0.005316,0.002337,0.002337,-0.002337,1.0,Up
2025-10-20 08:20:47.602642,100.999397,0.006459,0.002282,0.002282,-0.002282,1.0,Up


## Interactive timeline (Plotly)

An interactive chart lets viewers hover to see the exact price, future return and the label. This is useful when sharing the notebook with stakeholders or exporting to HTML.

In [ ]:
# Interactive Plotly timeline
plot_df = df.tail(300).reset_index()
plot_df['TargetColor'] = plot_df['Target'].map({0: 'red', 1: 'green', 2: 'gray'})
# Convert return-based thresholds into price-space thresholds for plotting
plot_df['UpperPrice'] = plot_df['Close'] * (1 + plot_df['UpperThreshold'])
plot_df['LowerPrice'] = plot_df['Close'] * (1 + plot_df['LowerThreshold'])
# Create arrow text showing direction and magnitude (▲ up, ▼ down, → sideways)
def arrow_label(fr, up_thr, low_thr):
    if fr > up_thr:
        return f'▲ {fr:.2%}'
    if fr < low_thr:
        return f'▼ {abs(fr):.2%}'
    return f'→ {fr:.2%}'
plot_df['ArrowText'] = plot_df.apply(lambda r: arrow_label(r['FutureReturn'], r['UpperThreshold'], r['LowerThreshold']), axis=1)
fig = go.Figure()
# Price line
fig.add_trace(go.Scatter(x=plot_df['index'], y=plot_df['Close'], mode='lines', name='Close', yaxis='y1'))
# Threshold lines (dashed)
fig.add_trace(go.Scatter(x=plot_df['index'], y=plot_df['UpperPrice'], mode='lines', name='Upper threshold',
                         line=dict(dash='dash', color='green'), yaxis='y1'))
fig.add_trace(go.Scatter(x=plot_df['index'], y=plot_df['LowerPrice'], mode='lines', name='Lower threshold',
                         line=dict(dash='dash', color='red'), yaxis='y1'))
# Markers with hover info
fig.add_trace(go.Scatter(x=plot_df['index'], y=plot_df['Close'], mode='markers',
                         marker=dict(color=plot_df['TargetColor'], size=8),
                         hovertemplate='Time: %{x}<br>Price: %{y:.4f}<br>Future ret: %{customdata[0]:.4f}<br>Label: %{customdata[1]}',
                         customdata=plot_df[['FutureReturn','TargetLabel']].values,
                         name='Labels', yaxis='y1'))
# Arrow text annotations split by class so colors are controlled
up_df = plot_df[plot_df['Target'] == 1]
down_df = plot_df[plot_df['Target'] == 0]
side_df = plot_df[plot_df['Target'] == 2]
fig.add_trace(go.Scatter(x=up_df['index'], y=up_df['Close'], mode='text', text=up_df['ArrowText'],
                         textposition='top center', textfont=dict(size=10, color='green'), showlegend=False, yaxis='y1'))
fig.add_trace(go.Scatter(x=down_df['index'], y=down_df['Close'], mode='text', text=down_df['ArrowText'],
                         textposition='top center', textfont=dict(size=10, color='red'), showlegend=False, yaxis='y1'))
fig.add_trace(go.Scatter(x=side_df['index'], y=side_df['Close'], mode='text', text=side_df['ArrowText'],
                         textposition='top center', textfont=dict(size=10, color='gray'), showlegend=False, yaxis='y1'))
# FutureReturn as price-space bars anchored at the Close price (start at the point and extend up/down)
plot_df['EndPrice'] = plot_df['Close'] * (1 + plot_df['FutureReturn'])
# Bar height (delta) = EndPrice - Close so bars start at the point and extend up/down by the move
plot_df['DeltaPrice'] = plot_df['EndPrice'] - plot_df['Close']
# Color bars by target class to match markers
bar_colors = plot_df['Target'].map({0: 'rgba(255,0,0,0.35)', 1: 'rgba(0,128,0,0.35)', 2: 'rgba(128,128,128,0.25)'})
# Use a 1-hour width for bars (ms) to align with hourly data; adjust if your data frequency differs
one_hour_ms = 60 * 60 * 1000
fig.add_trace(go.Bar(x=plot_df['index'], base=plot_df['Close'], y=plot_df['DeltaPrice'], marker_color=bar_colors, name='FutureReturn (price)', width=one_hour_ms, hovertemplate='Time: %{x}<br>Start price: %{base:.4f}<br>End price: %{customdata[0]:.4f}<br>Future ret: %{customdata[1]:.2%}', customdata=np.vstack([plot_df['EndPrice'], plot_df['FutureReturn']]).T, showlegend=True, yaxis='y1'))
fig.update_layout(title='Interactive price timeline with target labels, thresholds and future-return arrows', xaxis_title='Time', yaxis=dict(title='Price', side='left', showgrid=False), height=560)
fig.show()

### Notes for sharing
- Add the short plain-language paragraph (above) when presenting the plots.
- Use the static plot for slide decks and the interactive Plotly for exploratory demonstrations.
- You can adjust LOOKBACK, HORIZON and MULTIPLIER to make the target more/less sensitive to price moves.